# 02 ,  Extraccion y alineacion inicial del corpus paralelo

**TFM Inga-Espanol.** Este notebook produce la primera version del corpus paralelo a partir de dos fuentes primarias:
- El Nuevo Testamento en Inga (Wycliffe, 2012) alineado por referencia canonica.
- Las narrativas de Antihua Pacay (Jamioy, 1985) alineadas por bloques bilingues.

**Salidas:**
- `datos/nt_inga_alineado.jsonl`
- `datos/antihua_pacay_alineado.jsonl`

## Imports y rutas

In [1]:
from pathlib import Path
import json, re
from collections import Counter

REPO_OCR = Path('/Users/william-santos/Documents/UNIR/tfm/datos/ocr/inga-kichwa')
EXT_OCR  = Path('/Users/william-santos/Documents/UNIR/mistral-ocr-extractor/output/inga-kichwa')
OCR_ROOT = REPO_OCR if REPO_OCR.exists() else EXT_OCR
print('OCR_ROOT =', OCR_ROOT)

DATA_OUT = Path('/Users/william-santos/Documents/UNIR/tfm/datos')
DATA_OUT.mkdir(parents=True, exist_ok=True)

NT_PATH = OCR_ROOT / '00-WNTinb-web.md'
AP_PATH = OCR_ROOT / 'antihua-pacay.md'
print('NT:', NT_PATH.exists(), '|', 'Antihua Pacay:', AP_PATH.exists())

OCR_ROOT = /Users/william-santos/Documents/UNIR/tfm/datos/ocr/inga-kichwa
NT: True | Antihua Pacay: True


## Parte A. Segmentacion del Nuevo Testamento Inga por libro y versiculo

El OCR del NT preserva marcadores de pagina (`<!-- Page N -->`) y los numeros de versiculo al inicio de cada renglon. Aqui detectamos libros por los headers Markdown (niveles 1 y 2), capitulos por numeros de dos digitos tras un header, y versiculos por numeros al comienzo de linea.

In [2]:
nt_text = NT_PATH.read_text(encoding='utf-8')

# Estrategia simplificada: segmentar todo el texto por versiculos numerados
# y conservar pares (numero_versiculo, texto) ignorando de momento la asignacion a libro/capitulo.
verse_splits = re.split(r'(?m)^(\d{1,3})\s+', nt_text)
# verse_splits = [preamble, num1, texto1, num2, texto2, ...]
pairs = []
for i in range(1, len(verse_splits) - 1, 2):
    num = verse_splits[i]
    raw = verse_splits[i + 1]
    # Cortar el texto antes del siguiente marcador de estructura
    cut = re.split(r'(<!-- Page \d+ -->|^#+ |\n\n\n)', raw, maxsplit=1)
    text = cut[0].strip()
    text = re.sub(r'\s+', ' ', text)
    if 3 < len(text.split()) < 120 and not text.startswith('<!--'):
        pairs.append({'verso_num': int(num), 'texto_inga': text})

print(f'Pares (verso, texto) candidatos: {len(pairs)}')
for p in pairs[:5]:
    print(f"  v.{p['verso_num']:>3} | {p['texto_inga'][:80]}")

Pares (verso, texto) candidatos: 3093
  v.  1 | Korintopi Cristowa tukuskakunata ... 1 Korintopi (1 Ko) ... 361
  v.  2 | Korintopi Cristowa tukuskakunata ... 2 Korintopi (2 Ko) ... 390 Galasiapi Cristo
  v.  1 | Tesalonikapi Cristowa tukuskakunata ... 1 Tesalonikapi (1 Ts) ... 447
  v.  2 | Tesalonikapi Cristowa tukuskakunata ... 2 Tesalonikapi (2 Ts) ... 453
  v.  1 | Timoteota ... 1 Timoteota (1 Ti) ... 457


## Guardar el NT Inga segmentado como corpus monolingue ordenado

En esta Entrega 1 la alineacion canonica completa con un NT espanol de dominio publico se difiere a la Entrega 2 (requiere mapeo libro por libro y version RV 1909 o analoga). Se guarda el NT Inga segmentado con indice de orden para uso posterior.

In [3]:
out_nt = DATA_OUT / 'nt_inga_alineado.jsonl'
with open(out_nt, 'w', encoding='utf-8') as f:
    for i, p in enumerate(pairs):
        f.write(json.dumps({'idx': i, **p, 'texto_es': None}, ensure_ascii=False) + '\n')
print(f'Segmentos NT Inga guardados: {len(pairs)} en {out_nt}')

Segmentos NT Inga guardados: 3093 en /Users/william-santos/Documents/UNIR/tfm/datos/nt_inga_alineado.jsonl


## Parte B. Antihua Pacay ,  deteccion de bloques por idioma y alineacion

Antihua Pacay presenta la estructura: bloque en Inga, luego bloque en espanol, y asi sucesivamente. Se usa un detector heuristico basado en la densidad de sufijos aglutinantes tipicos del Inga.

In [4]:
ap_text = AP_PATH.read_text(encoding='utf-8')

# Segmentar por parrafos separados por dos saltos de linea o mas
paragraphs = [p.strip() for p in re.split(r'\n\s*\n', ap_text) if p.strip()]
# Filtrar comentarios OCR y encabezados
paragraphs = [p for p in paragraphs if not p.startswith('<!--') and not p.startswith('#')
              and '![img' not in p and len(p.split()) >= 4]
print(f'Parrafos textuales detectados: {len(paragraphs)}')

Parrafos textuales detectados: 284


In [5]:
# Detector: densidad de sufijos aglutinantes del Inga
INGA_SUFFIXES = ('cuna', 'manda', 'spa', 'mi', 'si', 'pi', 'ta', 'huan', 'pac', 'pura',
                 'rispa', 'nchi', 'ngapa', 'guna', 'yucado', 'sca', 'sinchu', 'ngi')
ES_STOPWORDS = {'de', 'la', 'el', 'los', 'las', 'un', 'una', 'que', 'y', 'en', 'con',
                'por', 'para', 'es', 'se', 'del', 'al', 'como', 'pero', 'su', 'sus',
                'este', 'esta', 'lo', 'le', 'les'}

def score_inga(text):
    tokens = [t.lower().strip('.,;:!?()"\u00bf\u00a1') for t in text.split()]
    tokens = [t for t in tokens if t]
    if not tokens:
        return 0.0
    inga_hits = sum(1 for t in tokens if any(t.endswith(s) for s in INGA_SUFFIXES))
    es_hits = sum(1 for t in tokens if t in ES_STOPWORDS)
    return (inga_hits - es_hits) / len(tokens)

labeled = []
for p in paragraphs:
    sc = score_inga(p)
    lang = 'inga' if sc > 0.02 else ('es' if sc < -0.05 else 'mixed')
    labeled.append({'lang': lang, 'score': round(sc, 3), 'text': p})

from collections import Counter
print('Distribucion por idioma detectado:', Counter(x['lang'] for x in labeled))

Distribucion por idioma detectado: Counter({'inga': 144, 'es': 136, 'mixed': 4})


In [6]:
# Emparejar bloques consecutivos Inga -> ES cuando aparezcan en ese orden
pares_ap = []
i = 0
while i < len(labeled) - 1:
    if labeled[i]['lang'] == 'inga' and labeled[i + 1]['lang'] == 'es':
        pares_ap.append({
            'idx': len(pares_ap),
            'texto_inga': labeled[i]['text'],
            'texto_es': labeled[i + 1]['text'],
        })
        i += 2
    else:
        i += 1

print(f'Pares bloque-a-bloque detectados: {len(pares_ap)}')
for p in pares_ap[:3]:
    print(f"  INGA: {p['texto_inga'][:80]}...")
    print(f"  ES  : {p['texto_es'][:80]}...")
    print()

out_ap = DATA_OUT / 'antihua_pacay_alineado.jsonl'
with open(out_ap, 'w', encoding='utf-8') as f:
    for p in pares_ap:
        f.write(json.dumps(p, ensure_ascii=False) + '\n')
print(f'Guardado: {out_ap}')

Pares bloque-a-bloque detectados: 4
  INGA: Antihua Pacay Gentecunapa Parlocuna...
  ES  : Otilia Jamioy Yanangona de Peña...

  INGA: Y ni pay manima pudihúrac, niscacuna: -Más suma atún hachayug canmi picudu....
  ES  : Paipas samuspa cuchuhura, cahuanacúgpic, limpios del todo trozado can chi sáchac...

  INGA: 1. Huahuacuna huacangapa callarígpic, čímata samucudur?
2. Huahuapagma caillayás...
  ES  : Hemos elaborado este librito para que tú también sepas por qué nuestra gente vin...

Guardado: /Users/william-santos/Documents/UNIR/tfm/datos/antihua_pacay_alineado.jsonl


## Resumen de salida

In [7]:
import json
stats_path = DATA_OUT / 'estadisticas_corpus.json'
stats = json.loads(stats_path.read_text()) if stats_path.exists() else {}
stats['nt_segmentos_inga'] = len(pairs)
stats['antihua_pacay_parrafos_detectados'] = len(paragraphs)
stats['antihua_pacay_pares_bloque_a_bloque'] = len(pares_ap)
stats_path.write_text(json.dumps(stats, ensure_ascii=False, indent=2))
print(json.dumps(stats, ensure_ascii=False, indent=2))

{
  "recursos": {
    "diccionario": {
      "chars": 469272,
      "words": 72212,
      "lines": 19152,
      "pages": 177
    },
    "gramatica_pedagogica": {
      "chars": 326093,
      "words": 54172,
      "lines": 7434,
      "pages": 228
    },
    "apendice_morfosintactico": {
      "chars": 31344,
      "words": 5130,
      "lines": 951,
      "pages": 26
    },
    "nuevo_testamento": {
      "chars": 1474063,
      "words": 178385,
      "lines": 17759,
      "pages": 595
    },
    "antihua_pacay": {
      "chars": 65311,
      "words": 9767,
      "lines": 1302,
      "pages": 68
    }
  },
  "total_paginas_ocr": 1094,
  "diccionario_entradas_detectadas": 817,
  "nt_versiculos_aproximados": 2837,
  "nt_versiculos_longitud_valida": 2791,
  "nt_longitud_media_palabras_inga": 32.75,
  "nt_segmentos_inga": 3093,
  "antihua_pacay_parrafos_detectados": 284,
  "antihua_pacay_pares_bloque_a_bloque": 4
}


## Notas metodologicas

- El NT Inga queda segmentado y listo para alineacion canonica con Reina-Valera 1909 en la Entrega 2.
- Antihua Pacay produjo un primer conjunto de pares bloque-a-bloque por alineacion heuristica; la segmentacion intra-bloque a nivel de oracion se refina en la siguiente iteracion.
- Ambos archivos JSONL alimentan el corpus paralelo de la Fase 2 y las evaluaciones de la Fase 6.